In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Concatenação dos datasets

In [ ]:
from pathlib import Path
import pandas as pd

CURATED_DIR = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated"
)

FILTERED_DATABASE_FILE = (
    CURATED_DIR / "banco_python_ensino_limpo.parquet"
)

PYTHON_DOCS_FILE = (
    CURATED_DIR / "python_docs_sem_duplicatas.parquet"
)

OUTPUT_FILE = (
    CURATED_DIR / "banco_python_concatenado.parquet"
)

print("=" * 80)
print("CONCATENAÇÃO DOS DATASETS")
print("=" * 80)

print("\nCarregando banco filtrado...")

df_database = pd.read_parquet(FILTERED_DATABASE_FILE)

print(
    f"Registros originais: {len(df_database):,}"
)

print("\nSources originais:")
print(
    df_database["source"]
    .value_counts()
    .to_string()
)

RETAINED_SOURCES = [
    "glaive_python",
    "faq",
    "dataset_python_qa"
]

df_database = df_database[
    df_database["source"].isin(RETAINED_SOURCES)
].copy()

df_database = df_database.reset_index(drop=True)

print("\n" + "=" * 80)
print("BANCO PRINCIPAL FILTRADO")
print("=" * 80)

print(
    f"Registros mantidos: {len(df_database):,}"
)

print("\nSources mantidas:")

print(
    df_database["source"]
    .value_counts()
    .to_string()
)

print("\n" + "=" * 80)
print("CARREGANDO PYTHON DOCS")
print("=" * 80)

df_docs = pd.read_parquet(PYTHON_DOCS_FILE)

print(
    f"Registros python_docs: {len(df_docs):,}"
)

print("\nSources encontrados:")

print(
    df_docs["source"]
    .value_counts()
    .to_string()
)

df_docs = df_docs[
    df_docs["source"] == "python_docs"
].copy()

df_docs = df_docs.reset_index(drop=True)

print(
    f"\nRegistros python_docs mantidos: "
    f"{len(df_docs):,}"
)

REQUIRED_COLUMNS = [
    "question",
    "answer",
    "code",
    "source",
    "language"
]

for dataset_name, dataframe in [
    ("banco principal", df_database),
    ("python_docs", df_docs)
]:

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Colunas ausentes em {dataset_name}: {missing_columns}"
        )

df_database = df_database[
    REQUIRED_COLUMNS
].copy()

df_docs = df_docs[
    REQUIRED_COLUMNS
].copy()

print("\n" + "=" * 80)
print("CONCATENANDO")
print("=" * 80)

df_final = pd.concat(
    [
        df_docs,
        df_database
    ],
    ignore_index=True
)

df_final = df_final.reset_index(drop=True)

print(
    f"Total após concatenação: "
    f"{len(df_final):,}"
)

print("\n" + "=" * 80)
print("DISTRIBUIÇÃO FINAL DOS SOURCES")
print("=" * 80)

print(
    df_final["source"]
    .value_counts()
    .to_string()
)

EXPECTED_SOURCES = {
    "python_docs",
    "glaive_python",
    "faq",
    "dataset_python_qa"
}

FOUND_SOURCES = set(
    df_final["source"].dropna().unique()
)

UNEXPECTED_SOURCES = (
    FOUND_SOURCES - EXPECTED_SOURCES
)

if UNEXPECTED_SOURCES:
    raise ValueError(
        f"Sources indevidos encontrados: "
        f"{UNEXPECTED_SOURCES}"
    )

df_final.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 80)
print("CONCATENAÇÃO CONCLUÍDA")
print("=" * 80)

print(
    f"Arquivo salvo em:\n{OUTPUT_FILE}"
)

print(
    f"Total de registros: {len(df_final):,}"
)

Verificação da estrutura do dataset concatenado

In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_concatenado.parquet"
]

for file in files:
    print("=" * 100)
    print(f"Arquivo: {file}")

    df = pd.read_parquet(file)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nPrimeiras linhas:")
    print(df.head())

    print("\nQuantidade de registros:")
    print(len(df))

    print("\nValores nulos:")
    print(df.isnull().sum())

    print("\nInformações do DataFrame:")
    print(df.info())

Verificação da ordem dos sources

In [ ]:
import pandas as pd

INPUT_FILE = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_concatenado.parquet"
)

df = pd.read_parquet(INPUT_FILE)

print("=" * 80)
print("VERIFICAÇÃO DA ORDEM DOS SOURCES")
print("=" * 80)

print(f"Total de registros: {len(df):,}")

print("\n" + "=" * 80)
print("QUANTIDADE POR SOURCE")
print("=" * 80)

print(
    df["source"]
    .value_counts()
    .to_string()
)

print("\n" + "=" * 80)
print("ORDEM DE APARIÇÃO DOS SOURCES")
print("=" * 80)

source_order = (
    df["source"]
    .drop_duplicates()
    .tolist()
)

for index, source in enumerate(source_order, start=1):
    print(f"{index}. {source}")

print("\n" + "=" * 80)
print("INTERVALOS DE CADA SOURCE")
print("=" * 80)

source_changes = (
    df["source"]
    != df["source"].shift()
)

source_blocks = df[source_changes].copy()

source_blocks["end_index"] = (
    source_blocks.index.to_series()
    .shift(-1)
    .fillna(len(df))
    .astype(int) - 1
)

for _, row in source_blocks.iterrows():

    start_index = int(row.name)
    end_index = int(row["end_index"])
    source = row["source"]

    record_count = end_index - start_index + 1

    print(
        f"{source}: "
        f"{start_index:,} → {end_index:,} "
        f"({record_count:,} registros)"
    )

print("\n" + "=" * 80)
print("PRIMEIROS 10 REGISTROS")
print("=" * 80)

print(
    df[
        ["question", "source"]
    ]
    .head(10)
    .to_string(index=True)
)

print("\n" + "=" * 80)
print("ÚLTIMOS 10 REGISTROS")
print("=" * 80)

print(
    df[
        ["question", "source"]
    ]
    .tail(10)
    .to_string(index=True)
)